# Example 1.2.3 / 2: Symbolic Parameters

Companion notebook for **1.2.3 Symbolic and Logical Parameters** in the AMPL lab Documentation, section "Symbolic Parameters". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Symbolic Parameters

The restrictions `in INTER` and `<> entr` catch mistakes in data read from a data file, as for
numeric parameters. A value assigned from Python is not tested the same way: here the exit
equals the entrance, and amplpy accepts it and returns it without an error:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set INTER;
param entr symbolic in INTER;
param exit symbolic in INTER, <> entr;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["INTER"] = ["a", "b", "c", "d", "e", "f", "g"]
ampl.param["entr"] = "a"
ampl.param["exit"] = "a"

In [5]:
# run.py
print(ampl.param["exit"].value())

a


### Pure AMPL: data.dat and model.run

In pure AMPL, with the same `model.mod`, the data file's value is tested and rejected:

In [6]:
%%writefile data.dat
set INTER := a b c d e f g;
param entr := a;
param exit := a;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display exit;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing symbolic param exit:
		failed check: param exit = a
			is not != entr;


exit = a

